In [ ]:
from IPython.display import Image, Markdown, display

display(Image(filename="Garbage Net Architecture.png"))
display(Markdown("<center><b>Figure 1: Garbage Net Architecture</b></center>"))

In [ ]:
from pathlib import Path
import os
import json
import warnings

PROJECT_ROOT = Path.cwd()
CACHE_ROOT = PROJECT_ROOT / ".notebook_cache"
(CACHE_ROOT / "matplotlib").mkdir(parents=True, exist_ok=True)
(CACHE_ROOT / "xdg").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_ROOT / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_ROOT / "xdg"))

import contextlib
import io
import numpy as np
import pandas as pd
with contextlib.redirect_stderr(io.StringIO()):
    import matplotlib.pyplot as plt
    import seaborn as sns
from PIL import Image as PILImage
from IPython.display import display, Markdown, SVG

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
sns.set_theme(style="whitegrid", context="notebook")

LABEL_ORDER = [
    "battery", "biological", "cardboard", "clothes", "glass",
    "metal", "paper", "plastic", "shoes", "trash"
]

REPORT_DIR = Path("data/processed/model_reports")
FIGURE_DIR = Path("data/processed/figures")

def read_csv(path: str | Path) -> pd.DataFrame:
    return pd.read_csv(Path(path))

def read_json(path: str | Path):
    with open(Path(path), "r", encoding="utf-8") as f:
        return json.load(f)

def show_table(df: pd.DataFrame, columns=None, n=None, sort_by=None, ascending=False):
    out = df.copy()
    if sort_by is not None:
        out = out.sort_values(sort_by, ascending=ascending)
    if columns is not None:
        out = out[columns]
    if n is not None:
        out = out.head(n)
    display(out.reset_index(drop=True))
    return out

def short_text(value, max_len=70):
    text = str(value)
    return text if len(text) <= max_len else text[:max_len - 3] + "..."

def bar_labels(ax, fmt="{:.3f}", rotation=0):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3, fontsize=9, rotation=rotation)

print("Notebook helpers ready.")


In [ ]:
class_dist = read_csv("data/processed/metadata/class_distribution.csv")
split_summary = read_csv("data/processed/splits/split_summary.csv")

split_table = split_summary.rename(columns={
    "label": "class",
    "total": "total",
    "train_count": "train",
    "validation_count": "validation",
    "test_count": "test",
})[["class", "total", "train", "validation", "test"]]

display(Markdown("**类别与固定划分统计**"))
display(split_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

order = class_dist.sort_values("count", ascending=False)["label"]
sns.barplot(data=class_dist, x="label", y="count", order=order, ax=axes[0], color="#4C78A8")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("")
axes[0].set_ylabel("Images")
axes[0].tick_params(axis="x", rotation=35)

split_plot = split_table.set_index("class").loc[LABEL_ORDER, ["train", "validation", "test"]]
split_plot.plot(kind="bar", stacked=True, ax=axes[1], color=["#4C78A8", "#F58518", "#54A24B"])
axes[1].set_title("Stratified Train / Validation / Test Split")
axes[1].set_xlabel("")
axes[1].set_ylabel("Images")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend(title="split")

plt.tight_layout()
plt.show()

totals = {
    "total_images": int(class_dist["count"].sum()),
    "train": int(split_summary["train_count"].sum()),
    "validation": int(split_summary["validation_count"].sum()),
    "test": int(split_summary["test_count"].sum()),
}
display(pd.DataFrame([totals]))


In [ ]:
feature_report = read_csv("data/processed/feature_reports/feature_extraction_report.csv")

main_features = feature_report[
    feature_report["feature_id"].isin([
        "F0_pixels_gray", "F0_pixels_rgb", "F1_color", "F2_texture",
        "F3_hog", "F4_shape", "F5_gist", "F6_bof"
    ])
].copy()
main_features["role"] = main_features["planned_role"].map(lambda x: short_text(x, 48))

display(Markdown("**特征组与维度**"))
display(main_features[[
    "feature_id", "role", "actual_dim", "train_shape",
    "validation_shape", "test_shape", "nan_count_total", "inf_count_total", "status"
]].reset_index(drop=True))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
sns.barplot(
    data=main_features.sort_values("actual_dim", ascending=False),
    x="feature_id", y="actual_dim", ax=ax, color="#4C78A8"
)
ax.set_title("Feature Dimensions")
ax.set_xlabel("")
ax.set_ylabel("Dimension")
ax.tick_params(axis="x", rotation=25)
bar_labels(ax, fmt="{:.0f}")
plt.tight_layout()
plt.show()

preview_path = Path("data/processed/previews/preprocessing_preview_grid.jpg")
if preview_path.exists():
    display(Markdown("**预处理抽样检查图**"))
    display(PILImage.open(preview_path))


In [ ]:
s2_fixed = read_csv(REPORT_DIR / "stage2_single_feature_baselines.csv")
s2_fixed = s2_fixed[s2_fixed["status"].eq("ok")].copy()

top_fixed = s2_fixed.sort_values("validation_macro_f1", ascending=False).head(10)
display(Markdown("**固定参数单特征 baseline Top 10**"))
display(top_fixed[[
    "feature_id", "model", "feature_dim", "validation_accuracy",
    "validation_macro_f1", "minority_recall_mean",
    "recall_trash", "recall_battery", "recall_biological"
]].reset_index(drop=True))

best_fixed_idx = s2_fixed.groupby("feature_id")["validation_macro_f1"].idxmax()
best_fixed = s2_fixed.loc[best_fixed_idx].sort_values("validation_macro_f1", ascending=False)
display(Markdown("**每组特征的最佳固定参数结果**"))
display(best_fixed[[
    "feature_id", "model", "feature_dim",
    "validation_macro_f1", "minority_recall_mean"
]].reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
sns.barplot(
    data=best_fixed,
    x="feature_id", y="validation_macro_f1",
    ax=axes[0], color="#4C78A8"
)
axes[0].set_title("Best Fixed Baseline by Feature")
axes[0].set_xlabel("")
axes[0].set_ylabel("Validation Macro-F1")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_ylim(0, max(0.65, best_fixed["validation_macro_f1"].max() + 0.05))
bar_labels(axes[0])

sns.scatterplot(
    data=s2_fixed,
    x="validation_macro_f1", y="minority_recall_mean",
    hue="feature_id", style="model", s=90, ax=axes[1]
)
axes[1].set_title("Macro-F1 vs Minority Recall")
axes[1].set_xlabel("Validation Macro-F1")
axes[1].set_ylabel("Minority Recall Mean")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
s2_tuned = read_csv(REPORT_DIR / "stage2_tuned_best_by_feature.csv")
s2_tuned["best_params_short"] = s2_tuned["best_params"].map(lambda x: short_text(x, 95))

display(Markdown("**单特征调参后的最佳结果**"))
display(s2_tuned[[
    "feature_id", "model", "feature_dim",
    "fixed_validation_macro_f1", "validation_macro_f1",
    "minority_recall_mean", "search_strategy",
    "best_params_short"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True))

plot_df = s2_tuned.sort_values("validation_macro_f1", ascending=False).copy()
x = np.arange(len(plot_df))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
axes[0].bar(x - width / 2, plot_df["fixed_validation_macro_f1"], width, label="fixed")
axes[0].bar(x + width / 2, plot_df["validation_macro_f1"], width, label="tuned")
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["feature_id"], rotation=25)
axes[0].set_title("Fixed vs Tuned Single-Feature Results")
axes[0].set_ylabel("Validation Macro-F1")
axes[0].legend()

sns.scatterplot(
    data=plot_df,
    x="validation_macro_f1", y="minority_recall_mean",
    hue="feature_id", s=120, ax=axes[1]
)
for _, row in plot_df.iterrows():
    axes[1].text(row["validation_macro_f1"] + 0.002, row["minority_recall_mean"], row["feature_id"], fontsize=9)
axes[1].set_title("Tuned Feature Trade-off")
axes[1].set_xlabel("Validation Macro-F1")
axes[1].set_ylabel("Minority Recall Mean")
axes[1].legend_.remove()

plt.tight_layout()
plt.show()


In [ ]:
s3_screen = read_csv(REPORT_DIR / "stage3_model_screening.csv")
s3_best_combo = read_csv(REPORT_DIR / "stage3_best_by_combo.csv")
s3_best_model = read_csv(REPORT_DIR / "stage3_best_by_model.csv")
s3_add = read_csv(REPORT_DIR / "stage3_feature_addition_summary.csv")

display(Markdown("**阶段三广筛 Top 10**"))
display(s3_screen.sort_values("validation_macro_f1", ascending=False)[[
    "combo_id", "feature_ids", "model", "feature_dim",
    "validation_accuracy", "validation_macro_f1", "minority_recall_mean"
]].head(10).reset_index(drop=True))

display(Markdown("**各模型族的最佳结果**"))
display(s3_best_model[[
    "model", "combo_id", "feature_ids", "validation_macro_f1", "minority_recall_mean"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True))

heat = s3_screen.pivot_table(
    index="combo_id", columns="model", values="validation_macro_f1", aggfunc="max"
)
row_order = s3_best_combo.sort_values("validation_macro_f1", ascending=False)["combo_id"]
heat = heat.loc[row_order]

fig, axes = plt.subplots(1, 2, figsize=(17, 6.2))
sns.heatmap(heat, cmap="YlGnBu", annot=True, fmt=".3f", linewidths=0.4, ax=axes[0])
axes[0].set_title("Stage 3 Model Screening Heatmap")
axes[0].set_xlabel("Model")
axes[0].set_ylabel("Feature Combination")

add_plot = s3_add.sort_values("rank", ascending=False)
axes[1].plot(add_plot["validation_macro_f1"], add_plot["feature_ids"], marker="o", label="macro-F1")
axes[1].plot(add_plot["minority_recall_mean"], add_plot["feature_ids"], marker="s", label="minority recall")
axes[1].set_title("Feature Fusion Path")
axes[1].set_xlabel("Score")
axes[1].set_ylabel("")
axes[1].legend()
axes[1].grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
s4_search = read_csv(REPORT_DIR / "stage4_boosting_tuning_search.csv")
s4_final = read_csv(REPORT_DIR / "stage4_boosting_tuning_final.csv")
s4_final["best_params_short"] = s4_final["best_params"].map(lambda x: short_text(x, 90))

display(Markdown("**强模型精调后的 validation 结果**"))
display(s4_final[[
    "candidate_id", "model", "combo_label", "sample_weight_mode",
    "stage3_macro_f1", "validation_macro_f1", "macro_f1_delta_vs_stage3",
    "minority_recall_mean", "best_params_short"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.stripplot(
    data=s4_search,
    x="model", y="inner_macro_f1",
    hue="candidate_id", dodge=True, ax=axes[0], size=7
)
axes[0].set_title("Inner Holdout Search Trials")
axes[0].set_xlabel("")
axes[0].set_ylabel("Inner Macro-F1")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

compare = s4_final.sort_values("validation_macro_f1", ascending=False).head(6).copy()
x = np.arange(len(compare))
width = 0.36
axes[1].bar(x - width / 2, compare["stage3_macro_f1"], width, label="stage3 baseline")
axes[1].bar(x + width / 2, compare["validation_macro_f1"], width, label="tuned")
axes[1].set_xticks(x)
axes[1].set_xticklabels(compare["candidate_id"], rotation=35, ha="right")
axes[1].set_title("Stage 3 Baseline vs Tuned Final")
axes[1].set_ylabel("Validation Macro-F1")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
s4_ab = read_csv(REPORT_DIR / "stage4_lgbm_ablation_imbalance.csv")
ablation = s4_ab[s4_ab["experiment_type"].eq("ablation")].copy()
imbalance = s4_ab[s4_ab["experiment_type"].eq("imbalance")].copy()

display(Markdown("**LightGBM 特征消融结果**"))
display(ablation[[
    "combo_id", "feature_dim_raw", "sample_weight_mode",
    "validation_macro_f1", "minority_recall_mean",
    "delta_macro_f1_vs_C11_none", "recall_trash", "recall_battery", "recall_biological"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True))

display(Markdown("**类别不平衡处理结果**"))
display(imbalance[[
    "combo_id", "sample_weight_mode", "validation_macro_f1",
    "minority_recall_mean", "recall_trash", "recall_battery", "recall_biological"
]].sort_values(["combo_id", "validation_macro_f1"], ascending=[True, False]).reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.9))
abl_plot = ablation.sort_values("validation_macro_f1", ascending=True)
axes[0].barh(abl_plot["combo_id"], abl_plot["validation_macro_f1"], color="#4C78A8", label="macro-F1")
axes[0].scatter(abl_plot["minority_recall_mean"], abl_plot["combo_id"], color="#F58518", label="minority recall", zorder=3)
axes[0].set_title("LightGBM Ablation")
axes[0].set_xlabel("Score")
axes[0].legend()

imb_plot = imbalance.sort_values(["combo_id", "sample_weight_mode"])
sns.barplot(
    data=imb_plot,
    x="combo_id", y="validation_macro_f1",
    hue="sample_weight_mode", ax=axes[1]
)
axes[1].set_title("Weight Mode Comparison")
axes[1].set_xlabel("")
axes[1].set_ylabel("Validation Macro-F1")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="weight")

plt.tight_layout()
plt.show()


In [ ]:
s4_stack_summary = read_json(REPORT_DIR / "stage4_stacking_summary.json")
s4_base = read_csv(REPORT_DIR / "stage4_stacking_base_summary.csv")
s4_stack = read_csv(REPORT_DIR / "stage4_stacking_results.csv")

display(Markdown("**Stacking base learners**"))
display(s4_base[[
    "base_id", "group", "model_type", "combo_id", "feature_dim_raw",
    "output_type", "validation_macro_f1", "validation_minority_recall_mean"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True))

display(Markdown("**Ensemble 方案对比**"))
stack_display = s4_stack[[
    "ensemble_id", "ensemble_type", "num_base_models", "num_meta_features",
    "meta_model", "meta_C", "meta_weight_mode",
    "validation_accuracy", "validation_macro_f1",
    "validation_minority_recall_mean", "delta_macro_f1_vs_stage4_best"
]].sort_values("validation_macro_f1", ascending=False).reset_index(drop=True)
display(stack_display)

best_stage4 = s4_stack_summary["stage4_best_reference"]["validation_macro_f1"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.9))
sns.barplot(
    data=s4_base.sort_values("validation_macro_f1", ascending=False),
    x="validation_macro_f1", y="base_id", ax=axes[0], color="#4C78A8"
)
axes[0].set_title("Base Learner Validation Macro-F1")
axes[0].set_xlabel("Validation Macro-F1")
axes[0].set_ylabel("")

sns.barplot(
    data=s4_stack.sort_values("validation_macro_f1", ascending=False),
    x="validation_macro_f1", y="ensemble_id", ax=axes[1], color="#54A24B"
)
axes[1].axvline(best_stage4, color="#E45756", linestyle="--", label="best single LGBM")
axes[1].set_title("Ensemble Validation Macro-F1")
axes[1].set_xlabel("Validation Macro-F1")
axes[1].set_ylabel("")
axes[1].legend()

plt.tight_layout()
plt.show()

best_ensemble = s4_stack_summary["best_ensemble"]
display(pd.DataFrame([{
    "Garbage Net validation macro-F1": best_ensemble["validation_macro_f1"],
    "Garbage Net validation minority recall": best_ensemble["validation_minority_recall_mean"],
    "base models": best_ensemble["num_base_models"],
    "meta features": best_ensemble["num_meta_features"],
    "meta learner": best_ensemble["meta_model"],
}]))


In [ ]:
final_summary = read_json(REPORT_DIR / "final_test_evaluation_summary.json")
final_models = read_csv(REPORT_DIR / "final_model_results.csv")
final_base = read_csv(REPORT_DIR / "final_base_results.csv")
final_per_class = read_csv(REPORT_DIR / "final_per_class_metrics.csv")
final_errors = read_csv(REPORT_DIR / "final_error_pairs.csv")

best = final_summary["best_final_model"]
key_metrics = pd.DataFrame([{
    "model_name": "Garbage Net",
    "model_id": best["model_id"],
    "test_accuracy": best["test_accuracy"],
    "test_macro_f1": best["test_macro_f1"],
    "test_weighted_f1": best["test_weighted_f1"],
    "test_minority_recall_mean": best["test_minority_recall_mean"],
    "test_recall_trash": best["test_recall_trash"],
    "test_recall_battery": best["test_recall_battery"],
    "test_recall_biological": best["test_recall_biological"],
}])
display(Markdown("**Garbage Net 最终 test 指标**"))
display(key_metrics)

display(Markdown("**最终 ensemble 方案 test 对比**"))
display(final_models[[
    "model_id", "model_type", "num_base_models",
    "test_accuracy", "test_macro_f1", "test_weighted_f1",
    "test_minority_recall_mean"
]].sort_values("test_macro_f1", ascending=False).reset_index(drop=True))

stack_class = final_per_class[
    (final_per_class["model_id"].eq("final_stack_small_plus_lgbm_anchor")) &
    (final_per_class["split"].eq("test"))
].copy()
display(Markdown("**Garbage Net 逐类别 test 指标**"))
display(stack_class[["class_name", "support", "precision", "recall", "f1"]].reset_index(drop=True))

display(Markdown("**主要错误对 Top 12**"))
display(final_errors.sort_values("count", ascending=False).head(12).reset_index(drop=True))

display(Markdown("**最终评价图表**"))
for svg_name in [
    "final_model_comparison.svg",
    "final_stack_confusion_matrix.svg",
    "final_stack_per_class_performance.svg",
]:
    svg_path = FIGURE_DIR / svg_name
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))


In [ ]:
error_pairs = read_csv(REPORT_DIR / "final_error_pairs.csv").sort_values("count", ascending=False)
error_examples = read_csv(REPORT_DIR / "final_error_examples.csv")

fig, ax = plt.subplots(figsize=(9, 5.2))
top_pairs = error_pairs.head(12).copy()
top_pairs["pair"] = top_pairs["true_label"] + " -> " + top_pairs["predicted_label"]
sns.barplot(data=top_pairs, x="count", y="pair", ax=ax, color="#E45756")
ax.set_title("Top Error Pairs")
ax.set_xlabel("Errors")
ax.set_ylabel("")
bar_labels(ax, fmt="{:.0f}")
plt.tight_layout()
plt.show()

display(Markdown("**高置信度误分类样本示例**"))
examples = error_examples.head(9).copy()
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, (_, row) in zip(axes.ravel(), examples.iterrows()):
    img_path = Path(row["path"])
    if img_path.exists():
        ax.imshow(PILImage.open(img_path))
    ax.set_title(
        f"true: {row['true_label']}\npred: {row['predicted_label']} ({row['confidence']:.2f})",
        fontsize=10
    )
    ax.axis("off")
for ax in axes.ravel()[len(examples):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
lit = read_csv(REPORT_DIR / "literature_traditional_results.csv")
lit_ok = lit[lit["status"].eq("ok")].copy()
lit_display = lit_ok[[
    "candidate_id", "source_short", "model_label", "feature_dim",
    "validation_accuracy", "validation_macro_f1",
    "validation_minority_recall_mean", "actual_train_seconds"
]].sort_values("validation_macro_f1", ascending=False)

display(Markdown("**传统文献候选的本地 validation 复刻结果**"))
display(lit_display.reset_index(drop=True))

comparison = pd.concat([
    lit_display[["candidate_id", "validation_macro_f1", "validation_minority_recall_mean"]]
    .rename(columns={"candidate_id": "method"}),
    pd.DataFrame([{
        "method": "Garbage Net (validation)",
        "validation_macro_f1": s4_stack_summary["best_ensemble"]["validation_macro_f1"],
        "validation_minority_recall_mean": s4_stack_summary["best_ensemble"]["validation_minority_recall_mean"],
    }])
], ignore_index=True).sort_values("validation_macro_f1", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5.2))
sns.barplot(data=comparison, x="validation_macro_f1", y="method", ax=ax, color="#4C78A8")
ax.set_title("Traditional References vs Garbage Net on Validation")
ax.set_xlabel("Validation Macro-F1")
ax.set_ylabel("")
plt.tight_layout()
plt.show()
